In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from skrub import TableReport
import numpy as np
from sksurv.nonparametric import kaplan_meier_estimator

N_LINES = 4
static_data_path = "/Users/malek/TheLAB/DynaSurv/data/model_entry_imputes_data_STATIC_no_staging.parquet"
dynamic_data_path = "/Users/malek/TheLAB/DynaSurv/data/model_entry_imputed_data_HR+HER2-_stable_types_categorized_V2.parquet"

In [ ]:
df_dynamic = pd.read_parquet(dynamic_data_path)
df_static = pd.read_parquet(static_data_path)

df_merge = df_dynamic.merge(
    df_static, on="usubjid", how="inner"
)
print(f"Number of unique patients before line filter:{df_merge["usubjid"].nunique()}")
df_merge = (
    df_merge.loc[df_merge["lineid"] <= N_LINES]
    .sort_values(by=["usubjid","lineid"])
    .reset_index(drop=True)
    .copy()
)
print(f"Number of unique patients after line filter:{df_merge['usubjid'].nunique()}")
print("expected to be equal")
# TableReport(df_merge, max_plot_columns=200)

In [ ]:
pat_per_line = df_merge.groupby("lineid")["usubjid"].nunique()
print(f"Number of unique patients per line:\n{pat_per_line}")

In [ ]:
outcome_col = "Y_onset_to_death"
event_col = "Y_global_death_status"
n_bins = 30
quantiles = np.linspace(0, 1, n_bins + 1)
bin_edges = df_merge.loc[df_merge[event_col], outcome_col].quantile(quantiles).values

fig, axes = plt.subplots(nrows=2, ncols=N_LINES, figsize=(20, 6), sharey=False)

for line in range(1, N_LINES + 1):
    ax1 = axes[0,line - 1]
    ax2 = axes[1,line - 1]
    line_data = df_merge[df_merge["lineid"] == line]
    line_data[outcome_col].hist(ax=ax1, bins=20)
    median_value = line_data[outcome_col].median()
    q99_value = line_data[outcome_col].quantile(0.99)
    max_value = line_data[outcome_col].max()
    ax1.axvline(median_value, color='red', linestyle='--', label=f"Median: {median_value:.2f}")
    ax1.axvline(q99_value, color='blue', linestyle=':', label=f"99th Percentile: {q99_value:.2f}")
    ax1.axvline(max_value, color='green', linestyle='-.', label=f"Max: {max_value:.2f}")

    ax1.set_title(f"Line {line}")
    ax1.legend()    
    # ax.set_ylim(0, 3200)

    ax1.set_xlabel("PF Survival (months)")
    ax1.set_ylabel("count")

    kmf = kaplan_meier_estimator(line_data[event_col], line_data[outcome_col])
    ax2.step(kmf[0], kmf[1], where="post")
    ax2.set_title(f"Kaplan-Meier Curve - Line {line}")
    ax2.set_xlabel("Overall Survival (months)")
    ax2.set_ylabel("Survival Probability")
    ax2.grid()

    for bin_start, bin_end in zip(bin_edges[:-1:2], bin_edges[1::2]):
        ax2.fill_betweenx(
            y=[0, 1],
            x1=bin_start,
            x2=bin_end,
            color="gray",
            alpha=0.5,
            step="post"
        )

# Common axis labels
# fig.supxlabel("Overall Survival (months)")
# fig.supylabel("Frequency")
plt.tight_layout()
plt.show()

In [ ]:
to_drop = set()
for line in range(1, N_LINES + 1):
    line_data = df_merge[df_merge["lineid"] == line]
    q99_value = line_data[outcome_col].quantile(0.99)
    pat_to_drop = line_data[line_data[outcome_col] >= q99_value]
    to_drop.update(pat_to_drop.index)
print(f"Number of patients to drop based on 99th percentile: {len(to_drop)}")

In [ ]:
outcome_col = "Y_onset_to_progression"
event_col = "Y_progression"
n_bins = 30
quantiles = np.linspace(0, 1, n_bins + 1)
bin_edges = df_merge.loc[df_merge[event_col], outcome_col].quantile(quantiles).values


fig, axes = plt.subplots(nrows=2, ncols=N_LINES, figsize=(20, 6), sharey=False)

for line in range(1, N_LINES + 1):
    ax1 = axes[0,line - 1]
    ax2 = axes[1,line - 1]
    line_data = df_merge[df_merge["lineid"] == line]
    line_data[outcome_col].hist(ax=ax1, bins=20)
    median_value = line_data[outcome_col].median()
    q99_value = line_data[outcome_col].quantile(0.99)
    max_value = line_data[outcome_col].max()
    ax1.axvline(median_value, color='red', linestyle='--', label=f"Median: {median_value:.2f}")
    ax1.axvline(q99_value, color='blue', linestyle=':', label=f"99th Percentile: {q99_value:.2f}")
    ax1.axvline(max_value, color='green', linestyle='-.', label=f"Max: {max_value:.2f}")

    ax1.set_title(f"Line {line}")
    ax1.legend()    
    # ax.set_ylim(0, 3200)

    ax1.set_xlabel("PF Survival (months)")
    ax1.set_ylabel("count")

    kmf = kaplan_meier_estimator(line_data[event_col], line_data[outcome_col])
    ax2.step(kmf[0], kmf[1], where="post")
    ax2.set_title(f"Kaplan-Meier Curve - Line {line}")
    ax2.set_xlabel("PF Survival (months)")
    ax2.set_ylabel("Survival Probability")
    ax2.grid()


    for bin_start, bin_end in zip(bin_edges[:-1:2], bin_edges[1::2]):
        ax2.fill_betweenx(
            y=[0, 1],
            x1=bin_start,
            x2=bin_end,
            color="gray",
            alpha=0.5,
            step="post"
        )

# Common axis labels
# fig.supxlabel("Overall Survival (months)")
# fig.supylabel("Frequency")
plt.tight_layout()
plt.show()